#### **Enterprise Star Schema Design + Advanced SQL ETL (MySQL)**

**Goal**: Build a professional, scalable data warehouse model suitable for enterprise performance reporting at Mastercard Foundation.

#### Imports & Database Connection

In [2]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import mysql.connector
from mysql.connector import Error

load_dotenv()  # Load .env file

# Database credentials from .env
DB_CONFIG = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'user': os.getenv('MYSQL_USER'),
    'password': os.getenv('MYSQL_PASSWORD'),
    'database': os.getenv('MYSQL_DATABASE', 'youth_employment_db')
}

# Connect to MySQL
try:
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()
    print("Successfully connected to MySQL")
except Error as e:
    print(f"Error: {e}")

Successfully connected to MySQL


#### Create Database & Use It

In [3]:
cursor.execute("CREATE DATABASE IF NOT EXISTS youth_employment_db")
cursor.execute("USE youth_employment_db")
print("Database ready")

Database ready


#### Create Star Schema (Dimension Tables)

In [4]:
# Dimension: Country
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_country (
    country_key INT AUTO_INCREMENT PRIMARY KEY,
    country_name VARCHAR(50),
    region VARCHAR(50),
    inserted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

# Dimension: Sector
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_sector (
    sector_key INT AUTO_INCREMENT PRIMARY KEY,
    sector_name VARCHAR(100),
    inserted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

# Dimension: Program
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_program (
    program_key INT AUTO_INCREMENT PRIMARY KEY,
    program_name VARCHAR(150),
    country_name VARCHAR(50),
    inserted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

print("Dimension tables created")

Dimension tables created


#### Create Fact Table

In [5]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_program_performance (
    participant_id VARCHAR(20) PRIMARY KEY,
    country_key INT,
    sector_key INT,
    program_key INT,
    age INT,
    gender VARCHAR(10),
    education_level VARCHAR(50),
    enrollment_date DATE,
    completion_status VARCHAR(30),
    placed TINYINT,
    placement_date DATE,
    employment_type VARCHAR(50),
    monthly_income_usd DECIMAL(10,2),
    still_employed_6m TINYINT,
    followup_6m_date DATE,
    inserted_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (country_key) REFERENCES dim_country(country_key),
    FOREIGN KEY (sector_key) REFERENCES dim_sector(sector_key)
)
""")

print("Fact table created with proper relationships")

Fact table created with proper relationships


#### Load Data from CSV & Populate Tables



In [6]:
# Load merged data
base = pd.read_csv('../data/processed/merged_analytical_base.csv')

# Populate dim_country
countries = base['country'].unique()
country_data = [(c, 'East Africa' if c in ['Kenya','Rwanda','Ethiopia'] else 'West Africa') for c in countries]

cursor.executemany("""
    INSERT IGNORE INTO dim_country (country_name, region) 
    VALUES (%s, %s)
""", country_data)

# Populate dim_sector
sectors = base['sector'].unique()
cursor.executemany("INSERT IGNORE INTO dim_sector (sector_name) VALUES (%s)", [(s,) for s in sectors])

conn.commit()
print("Dimension tables populated")

Dimension tables populated


#### Populate Fact Table (Main ETL)

In [9]:
base = pd.read_csv('../data/processed/merged_analytical_base.csv')

# Mappings
country_map = {name: idx+1 for idx, name in enumerate(base['country'].unique())}
sector_map = {name: idx+1 for idx, name in enumerate(base['sector'].unique())}

fact_data = base.copy()

fact_data['country_key'] = fact_data['country'].map(country_map)
fact_data['sector_key'] = fact_data['sector'].map(sector_map)
fact_data['program_key'] = 1

# Columns to insert (15 columns - excluding inserted_at)
insert_cols = [
    'participant_id', 'country_key', 'sector_key', 'program_key', 'age', 
    'gender', 'education_level', 'enrollment_date', 'completion_status', 
    'placed', 'placement_date', 'employment_type', 'monthly_income_usd', 
    'still_employed_6m', 'followup_6m_date'
]

# Critical: Replace NaN with None for MySQL
insert_data = []
for _, row in fact_data[insert_cols].iterrows():
    clean_row = tuple(None if pd.isna(x) else x for x in row)
    insert_data.append(clean_row)

# Insert query
insert_query = """
    INSERT IGNORE INTO fact_program_performance 
    (participant_id, country_key, sector_key, program_key, age, gender,
     education_level, enrollment_date, completion_status, placed, 
     placement_date, employment_type, monthly_income_usd, 
     still_employed_6m, followup_6m_date)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.executemany(insert_query, insert_data)
conn.commit()

print(f"Fact table populated successfully with {len(insert_data):,} records")

Fact table populated successfully with 25,000 records


#### Populate program Table

In [11]:
base = pd.read_csv('../data/processed/merged_analytical_base.csv')

# Get unique programs with country
programs = base[['program_name', 'country']].drop_duplicates().dropna()

# Prepare data
program_data = [(row['program_name'], row['country']) for _, row in programs.iterrows()]

# Insert into dim_program
cursor.executemany("""
    INSERT IGNORE INTO dim_program (program_name, country_name) 
    VALUES (%s, %s)
""", program_data)

conn.commit()

# Verification
cursor.execute("SELECT COUNT(*) FROM dim_program")
count = cursor.fetchone()[0]
print(f"dim_program table populated with {count} programs")

# Show sample
cursor.execute("SELECT * FROM dim_program LIMIT 10")
print(pd.DataFrame(cursor.fetchall(), columns=['program_key', 'program_name', 'country_name', 'inserted_at']))

dim_program table populated with 10 programs
   program_key                            program_name country_name  \
0            1             Young Africa Works Ethiopia     Ethiopia   
1            2                Young Africa Works Ghana        Ghana   
2            3                Youth Skills for Tourism       Rwanda   
3            4                  Agri-Youth Empowerment      Nigeria   
4            5                  Digital Jobs for Youth        Kenya   
5            6                Young Africa Works Kenya        Kenya   
6            7        Youth Entrepreneurship Programme        Ghana   
7            8       Hanga Ahazaza (Create the Future)       Rwanda   
8            9  Skills for Employment and Productivity     Ethiopia   
9           10              Young Africa Works Nigeria      Nigeria   

          inserted_at  
0 2026-05-02 12:53:58  
1 2026-05-02 12:53:58  
2 2026-05-02 12:53:58  
3 2026-05-02 12:53:58  
4 2026-05-02 12:53:58  
5 2026-05-02 12:53:58  
6 202

#### Create Advanced Views (For Dashboards)



In [10]:
cursor.execute("""
CREATE OR REPLACE VIEW vw_enterprise_kpis AS
SELECT 
    c.country_name,
    s.sector_name,
    COUNT(*) as total_participants,
    AVG(placed) as placement_rate,
    AVG(CASE WHEN gender = 'Female' THEN placed END) as female_placement_rate,
    AVG(still_employed_6m) as retention_6m_rate,
    AVG(monthly_income_usd) as avg_monthly_income
FROM fact_program_performance f
JOIN dim_country c ON f.country_key = c.country_key
JOIN dim_sector s ON f.sector_key = s.sector_key
GROUP BY c.country_name, s.sector_name;
""")

print("Enterprise KPI View created")

Enterprise KPI View created


#### Test & Close

In [12]:
# Test
test_df = pd.read_sql("SELECT * FROM vw_enterprise_kpis LIMIT 5", conn)
print(test_df)

conn.close()
print("completed successfully!")

C:\Users\tohiba\AppData\Local\Temp\ipykernel_11192\2941148305.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql("SELECT * FROM vw_enterprise_kpis LIMIT 5", conn)


  country_name sector_name  total_participants  placement_rate  \
0     Ethiopia  Green Jobs                1357          0.7770   
1       Rwanda  Green Jobs                 907          0.7773   
2        Kenya  Green Jobs                1602          0.7720   
3        Ghana  Green Jobs                 949          0.7823   
4      Nigeria  Green Jobs                1472          0.8163   

   female_placement_rate  retention_6m_rate  avg_monthly_income  
0                 0.7772             0.6909          282.246305  
1                 0.7711             0.7041          286.988764  
2                 0.7767             0.6783          284.599156  
3                 0.7782             0.6649          278.419244  
4                 0.8099             0.6847          281.303609  
completed successfully!
